# Structural Instability in Big Five South African Banking Equities

This notebook reproduces the empirical analysis for Paper 1. imports, project paths, data preparation, feature construction, modelling, diagnostics, outputs and figures.

**Data note:** the IRESS dataset is proprietary and should not be uploaded to GitHub. Place the Excel file in the `data/` folder before running the notebook.

In [ ]:
# Import libraries

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    skew,
    kurtosis,
    jarque_bera,
    ks_2samp,
    levene,
    fligner
)

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except ImportError:
    display = print

In [ ]:
# Set project path

# This cell works whether the notebook is opened from the repository root or from the notebooks folder.
current_dir = Path.cwd()
project_path = current_dir.parent if current_dir.name == "notebooks" else current_dir

data_path = project_path / "data"
outputs_path = project_path / "outputs"
figures_path = project_path / "figures"

for folder in [data_path, outputs_path, figures_path]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project path:", project_path)
print("Data path:", data_path)
print("Outputs path:", outputs_path)
print("Figures path:", figures_path)

In [ ]:
# Define input file and bank column names

# Place your IRESS Excel file in the data folder and update this file name if necessary.
input_file = data_path / "IRESS_DATASET.xlsx"

bank_column_map = {
    "STANDARD BANK GROUP LTD": "Standard Bank",
    "FIRSTRAND LTD": "FirstRand",
    "ABSA GROUP LTD": "Absa",
    "NEDBANK GROUP LTD": "Nedbank",
    "CAPITEC BANK HOLDINGS LTD": "Capitec"
}

banks = list(bank_column_map.values())

print("Banks included in the study:")
print(banks)
print("
Expected input file:")
print(input_file)

In [ ]:
# Helper functions

def format_pvalue(p_value):
    """Format p-values for journal tables."""
    if pd.isna(p_value):
        return np.nan
    if p_value < 0.0001:
        return "< 0.0001"
    return round(float(p_value), 4)


def save_table(table, filename):
    """Save a table to both Excel and CSV formats."""
    excel_path = outputs_path / filename
    csv_path = excel_path.with_suffix(".csv")

    table.to_excel(excel_path, index=False)
    table.to_csv(csv_path, index=False)

    return excel_path


def save_figure(filename):
    """Save the active matplotlib figure as a high-resolution PNG."""
    figure_path = figures_path / filename
    plt.tight_layout()
    plt.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.show()
    return figure_path


def safe_break_string(break_dates):
    """Convert break dates to a semicolon-separated string."""
    if break_dates is None or len(break_dates) == 0:
        return ""
    return "; ".join(pd.to_datetime(date).strftime("%Y-%m-%d") for date in break_dates)


def parse_break_dates(break_string):
    """Convert a stored break-date string back to pandas timestamps."""
    if pd.isna(break_string) or str(break_string).strip() == "":
        return []
    return [pd.to_datetime(item.strip()) for item in str(break_string).replace(",", ";").split(";") if item.strip()]

In [ ]:
# Load and clean IRESS dataset

if not input_file.exists():
    raise FileNotFoundError(
        f"Input file not found: {input_file}
"
        "Place your IRESS Excel file in the data folder and name it IRESS_DATASET.xlsx."
    )

raw_data = pd.read_excel(input_file)
raw_data["Date"] = pd.to_datetime(raw_data["Date"])
raw_data = raw_data.sort_values("Date").reset_index(drop=True)

bank_prices = raw_data.rename(columns=bank_column_map)[["Date"] + banks].copy()

for bank in banks:
    bank_prices[bank] = pd.to_numeric(bank_prices[bank], errors="coerce")

print("Dataset preview:")
display(bank_prices.head())

print("
Date range:")
print(bank_prices["Date"].min().strftime("%Y-%m-%d"), "to", bank_prices["Date"].max().strftime("%Y-%m-%d"))

print("
Missing values by column:")
display(bank_prices.isna().sum())

In [ ]:
# Save cleaned price data

prices = bank_prices.set_index("Date")
prices = prices.dropna(how="all")

prices.to_csv(outputs_path / "bank_price_data_clean.csv")

print("Cleaned price data:")
display(prices.head())

In [ ]:
# Construct daily log returns and volatility proxies

returns = 100 * np.log(prices / prices.shift(1))
returns = returns.dropna()

absolute_returns = returns.abs()
squared_returns = returns ** 2

returns.to_csv(outputs_path / "bank_log_returns.csv")
absolute_returns.to_csv(outputs_path / "bank_absolute_returns.csv")
squared_returns.to_csv(outputs_path / "bank_squared_returns.csv")

print("Return data shape:", returns.shape)
display(returns.head())

In [ ]:
# Descriptive statistics and normality tests

summary_rows = []

for bank in banks:
    x = returns[bank].dropna()
    jb_result = jarque_bera(x)

    summary_rows.append({
        "Bank": bank,
        "Observations": len(x),
        "Mean": x.mean(),
        "Standard deviation": x.std(),
        "Minimum": x.min(),
        "Maximum": x.max(),
        "Skewness": skew(x),
        "Excess kurtosis": kurtosis(x, fisher=True),
        "Jarque-Bera statistic": jb_result.statistic,
        "Jarque-Bera p-value": jb_result.pvalue,
        "Jarque-Bera p-value formatted": format_pvalue(jb_result.pvalue)
    })

summary_table = pd.DataFrame(summary_rows)

numeric_cols = summary_table.select_dtypes(include="number").columns
summary_table[numeric_cols] = summary_table[numeric_cols].round(4)

print("Descriptive statistics:")
display(summary_table)

save_table(summary_table, "Table_1_descriptive_statistics.xlsx")

In [ ]:
# Plot individual price series

for bank in banks:
    plt.figure(figsize=(9, 4))
    plt.plot(prices.index, prices[bank], linewidth=1.0)
    plt.title(f"Daily Stock Prices: {bank}")
    plt.xlabel("Date")
    plt.ylabel("Stock Price")
    plt.grid(alpha=0.3)
    save_figure(f"Figure_1_price_{bank.replace(' ', '')}.png")

In [ ]:
# Plot individual daily log-return series

for bank in banks:
    plt.figure(figsize=(9, 4))
    plt.plot(returns.index, returns[bank], linewidth=0.7)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.title(f"Daily Log Returns: {bank}")
    plt.xlabel("Date")
    plt.ylabel("Log Return (%)")
    plt.grid(alpha=0.3)
    save_figure(f"Figure_2_log_returns_{bank.replace(' ', '')}.png")

In [ ]:
# Structural-break helper functions

def compute_sse_matrix(y, min_size):
    """Compute segment sum of squared errors for mean-shift break detection."""
    n = len(y)
    y = np.asarray(y, dtype=float)

    csum = np.concatenate([[0], np.cumsum(y)])
    csum2 = np.concatenate([[0], np.cumsum(y ** 2)])
    sse = np.full((n, n), np.inf)

    for start in range(n):
        min_end = start + min_size
        if min_end > n:
            continue

        ends = np.arange(min_end, n + 1)
        segment_sum = csum[ends] - csum[start]
        segment_sum2 = csum2[ends] - csum2[start]
        segment_length = ends - start

        values = segment_sum2 - (segment_sum ** 2) / segment_length
        sse[start, ends - 1] = values

    return sse


def multiple_break_detection(series, max_breaks=5, min_size=120):
    """Detect multiple structural breaks using dynamic programming and BIC selection."""
    y_series = series.dropna()
    y = y_series.values
    dates = y_series.index
    n = len(y)

    sse = compute_sse_matrix(y, min_size)
    all_models = []

    for m in range(max_breaks + 1):
        segments = m + 1
        dp = np.full((segments, n), np.inf)
        previous = np.full((segments, n), -1, dtype=int)

        for end in range(min_size - 1, n):
            dp[0, end] = sse[0, end]

        for k in range(1, segments):
            for end in range((k + 1) * min_size - 1, n):
                best_value = np.inf
                best_split = -1

                split_start = k * min_size - 1
                split_end = end - min_size

                for split in range(split_start, split_end + 1):
                    value = dp[k - 1, split] + sse[split + 1, end]
                    if value < best_value:
                        best_value = value
                        best_split = split

                dp[k, end] = best_value
                previous[k, end] = best_split

        total_sse = dp[segments - 1, n - 1]

        if not np.isfinite(total_sse) or total_sse <= 0:
            continue

        parameters = segments + m
        bic = n * np.log(total_sse / n) + parameters * np.log(n)

        break_indices = []
        k = segments - 1
        end = n - 1

        while k > 0:
            split = previous[k, end]
            break_indices.append(split + 1)
            end = split
            k -= 1

        break_indices = sorted(break_indices)
        break_dates = [dates[i] for i in break_indices]

        all_models.append({
            "Number of breaks": m,
            "SSE": total_sse,
            "BIC": bic,
            "Break indices": break_indices,
            "Break dates": break_dates
        })

    if len(all_models) == 0:
        return {
            "Number of breaks": 0,
            "SSE": np.nan,
            "BIC": np.nan,
            "Break indices": [],
            "Break dates": []
        }, []

    best_model = min(all_models, key=lambda item: item["BIC"])
    return best_model, all_models


def run_break_detection(dataset, series_label, max_breaks=5, min_size=120):
    """Run structural-break detection for all banks."""
    selected_rows = []
    bic_rows = []

    for bank in banks:
        print(f"Running break detection for {bank}: {series_label}")

        best_model, all_models = multiple_break_detection(
            dataset[bank],
            max_breaks=max_breaks,
            min_size=min_size
        )

        selected_rows.append({
            "Bank": bank,
            "Series": series_label,
            "Selected number of breaks": best_model["Number of breaks"],
            "Selected BIC": best_model["BIC"],
            "Break dates": safe_break_string(best_model["Break dates"])
        })

        for model in all_models:
            bic_rows.append({
                "Bank": bank,
                "Series": series_label,
                "Number of breaks": model["Number of breaks"],
                "SSE": model["SSE"],
                "BIC": model["BIC"],
                "Break dates": safe_break_string(model["Break dates"])
            })

    selected_table = pd.DataFrame(selected_rows)
    bic_table = pd.DataFrame(bic_rows)

    for table in [selected_table, bic_table]:
        numeric_cols = table.select_dtypes(include="number").columns
        table[numeric_cols] = table[numeric_cols].round(4)

    return selected_table, bic_table

In [ ]:
# Run structural-break detection

absolute_breaks_table, absolute_bic_table = run_break_detection(
    absolute_returns,
    series_label="Absolute returns",
    max_breaks=5,
    min_size=120
)

squared_breaks_table, squared_bic_table = run_break_detection(
    squared_returns,
    series_label="Squared returns",
    max_breaks=5,
    min_size=120
)

print("Absolute-return structural breaks:")
display(absolute_breaks_table)

print("Squared-return structural breaks:")
display(squared_breaks_table)

save_table(absolute_breaks_table, "Table_2_absolute_return_structural_breaks.xlsx")
save_table(squared_breaks_table, "Table_3_squared_return_structural_breaks.xlsx")
save_table(absolute_bic_table, "Appendix_absolute_return_BIC_comparison.xlsx")
save_table(squared_bic_table, "Appendix_squared_return_BIC_comparison.xlsx")

In [ ]:
# Optional mean-return break check

mean_breaks_table, mean_bic_table = run_break_detection(
    returns,
    series_label="Mean returns",
    max_breaks=5,
    min_size=120
)

print("Mean-return break check:")
display(mean_breaks_table)

save_table(mean_breaks_table, "Appendix_mean_return_break_check.xlsx")
save_table(mean_bic_table, "Appendix_mean_return_BIC_comparison.xlsx")

In [ ]:
# Plot structural breaks for each bank

def extract_break_dates_from_table(break_table, bank):
    row = break_table[break_table["Bank"] == bank].iloc[0]
    return parse_break_dates(row["Break dates"])


def plot_breaks_for_each_bank(dataset, break_table, title_prefix, ylabel, filename_prefix):
    for bank in banks:
        break_dates = extract_break_dates_from_table(break_table, bank)

        plt.figure(figsize=(9, 4))
        plt.plot(dataset.index, dataset[bank], linewidth=0.8, label=bank)

        for break_date in break_dates:
            plt.axvline(break_date, linestyle="--", linewidth=1.0, alpha=0.8)

        plt.title(f"{title_prefix}: {bank}")
        plt.xlabel("Date")
        plt.ylabel(ylabel)
        plt.grid(alpha=0.3)
        plt.legend()
        save_figure(f"{filename_prefix}_{bank.replace(' ', '')}.png")


plot_breaks_for_each_bank(
    absolute_returns,
    absolute_breaks_table,
    title_prefix="Structural Breaks in Absolute Returns",
    ylabel="Absolute Return (%)",
    filename_prefix="absolute_return_breaks"
)

plot_breaks_for_each_bank(
    squared_returns,
    squared_breaks_table,
    title_prefix="Structural Breaks in Squared Returns",
    ylabel="Squared Return",
    filename_prefix="squared_return_breaks"
)

In [ ]:
# ICSS variance-break detection

def icss_breaks(series, critical_value=1.358, min_size=120):
    """Recursive ICSS-style detection of unconditional variance shifts."""
    y = series.dropna()
    dates = y.index
    values = y.values.astype(float)
    detected = []

    def _search(start, end):
        segment = values[start:end]
        n = len(segment)

        if n < 2 * min_size:
            return

        centered = segment - np.mean(segment)
        squares = centered ** 2
        total = squares.sum()

        if total <= 0:
            return

        cumulative = np.cumsum(squares)
        d_stat = cumulative / total - np.arange(1, n + 1) / n
        statistic = np.sqrt(n / 2) * np.abs(d_stat)
        local_break = int(np.argmax(statistic))
        max_stat = statistic[local_break]

        if max_stat > critical_value:
            global_break = start + local_break

            if global_break - start >= min_size and end - global_break >= min_size:
                detected.append(global_break)
                _search(start, global_break)
                _search(global_break, end)

    _search(0, len(values))
    detected = sorted(set(detected))
    return [dates[index] for index in detected]


icss_rows = []

for bank in banks:
    break_dates = icss_breaks(returns[bank], critical_value=1.358, min_size=120)

    icss_rows.append({
        "Bank": bank,
        "Number of ICSS variance breaks": len(break_dates),
        "ICSS break dates": safe_break_string(break_dates)
    })

icss_table = pd.DataFrame(icss_rows)

print("ICSS variance-break results:")
display(icss_table)

save_table(icss_table, "Table_4_ICSS_variance_breaks.xlsx")

In [ ]:
# Regime validation using KS, Fligner-Killeen and Levene tests

def create_regime_intervals(index, break_dates):
    """Create adjacent regime intervals from break dates."""
    if len(break_dates) == 0:
        return [(index.min(), index.max())]

    sorted_breaks = sorted(break_dates)
    dates = [index.min()] + sorted_breaks + [index.max()]
    return [(dates[i], dates[i + 1]) for i in range(len(dates) - 1)]


def regime_validation_tests(return_series, break_dates, source_series_label):
    """Compare adjacent return regimes created by detected break dates."""
    intervals = create_regime_intervals(return_series.index, break_dates)
    rows = []

    for i in range(len(intervals) - 1):
        start_1, end_1 = intervals[i]
        start_2, end_2 = intervals[i + 1]

        data1 = return_series[(return_series.index >= start_1) & (return_series.index <= end_1)]
        data2 = return_series[(return_series.index > start_2) & (return_series.index <= end_2)]

        if len(data1) < 20 or len(data2) < 20:
            continue

        ks_stat, ks_p = ks_2samp(data1, data2)
        fligner_stat, fligner_p = fligner(data1, data2)
        levene_stat, levene_p = levene(data1, data2, center="median")

        rows.append({
            "Break source": source_series_label,
            "Comparison": f"Regime {i + 1} vs Regime {i + 2}",
            "Regime 1 start": start_1.strftime("%Y-%m-%d"),
            "Regime 1 end": end_1.strftime("%Y-%m-%d"),
            "Regime 2 start": start_2.strftime("%Y-%m-%d"),
            "Regime 2 end": end_2.strftime("%Y-%m-%d"),
            "KS statistic": ks_stat,
            "KS p-value": ks_p,
            "KS p-value formatted": format_pvalue(ks_p),
            "KS decision": "Different distributions" if ks_p < 0.05 else "No significant distributional difference",
            "Fligner statistic": fligner_stat,
            "Fligner p-value": fligner_p,
            "Fligner p-value formatted": format_pvalue(fligner_p),
            "Fligner decision": "Different variances" if fligner_p < 0.05 else "No significant variance difference",
            "Levene statistic": levene_stat,
            "Levene p-value": levene_p,
            "Levene p-value formatted": format_pvalue(levene_p),
            "Levene decision": "Different variances" if levene_p < 0.05 else "No significant variance difference"
        })

    return pd.DataFrame(rows)


validation_tables = []

for bank in banks:
    for break_table, label in [
        (absolute_breaks_table, "Absolute-return breaks"),
        (squared_breaks_table, "Squared-return breaks")
    ]:
        break_dates = extract_break_dates_from_table(break_table, bank)
        temp = regime_validation_tests(returns[bank], break_dates, label)

        if not temp.empty:
            temp.insert(0, "Bank", bank)
            validation_tables.append(temp)

regime_validation_table = (
    pd.concat(validation_tables, ignore_index=True)
    if len(validation_tables) > 0
    else pd.DataFrame()
)

if not regime_validation_table.empty:
    numeric_cols = regime_validation_table.select_dtypes(include="number").columns
    regime_validation_table[numeric_cols] = regime_validation_table[numeric_cols].round(4)

print("Full regime-validation results:")
display(regime_validation_table)

save_table(regime_validation_table, "Appendix_full_regime_validation_tests.xlsx")

In [ ]:
# Compact regime-validation summary

validation_summary_rows = []

for bank in banks:
    bank_tests = regime_validation_table[regime_validation_table["Bank"] == bank] if not regime_validation_table.empty else pd.DataFrame()

    if bank_tests.empty:
        ks_count = 0
        variance_count = 0
        total_count = 0
    else:
        ks_count = (bank_tests["KS decision"] == "Different distributions").sum()
        variance_count = (
            (bank_tests["Fligner decision"] == "Different variances") |
            (bank_tests["Levene decision"] == "Different variances")
        ).sum()
        total_count = (
            (bank_tests["KS decision"] == "Different distributions") |
            (bank_tests["Fligner decision"] == "Different variances") |
            (bank_tests["Levene decision"] == "Different variances")
        ).sum()

    validation_summary_rows.append({
        "Bank": bank,
        "Significant distributional differences": int(ks_count),
        "Significant variance differences": int(variance_count),
        "Total significant adjacent-regime differences": int(total_count)
    })

validation_summary_table = pd.DataFrame(validation_summary_rows)

print("Compact regime-validation summary:")
display(validation_summary_table)

save_table(validation_summary_table, "Table_5_regime_validation_summary.xlsx")

In [ ]:
# Monthly clustering of detected breaks

def break_table_to_long_format(break_table, break_type, date_column="Break dates"):
    rows = []

    for _, row in break_table.iterrows():
        bank = row["Bank"]
        break_dates = parse_break_dates(row[date_column])

        for break_date in break_dates:
            rows.append({
                "Bank": bank,
                "Break type": break_type,
                "Break date": break_date,
                "Break month": break_date.to_period("M").strftime("%Y-%m"),
                "Break year": break_date.year
            })

    return pd.DataFrame(rows)


icss_long = icss_table.rename(columns={"ICSS break dates": "Break dates"})

all_break_dates = pd.concat(
    [
        break_table_to_long_format(absolute_breaks_table, "Absolute-return break"),
        break_table_to_long_format(squared_breaks_table, "Squared-return break"),
        break_table_to_long_format(icss_long, "ICSS variance break")
    ],
    ignore_index=True
)

if not all_break_dates.empty:
    monthly_break_clusters = (
        all_break_dates
        .groupby(["Break month", "Break type"])
        .agg(
            Number_of_breaks=("Break date", "count"),
            Banks=("Bank", lambda values: ", ".join(sorted(set(values))))
        )
        .reset_index()
        .sort_values(["Break month", "Break type"])
    )
else:
    monthly_break_clusters = pd.DataFrame(columns=["Break month", "Break type", "Number_of_breaks", "Banks"])

print("All detected break dates:")
display(all_break_dates)

print("Monthly break clusters:")
display(monthly_break_clusters)

save_table(all_break_dates, "Appendix_all_detected_break_dates.xlsx")
save_table(monthly_break_clusters, "Table_6_monthly_break_clusters.xlsx")

In [ ]:
# Composite Structural Instability Score

def minmax_scale(series):
    """Min-max scale a pandas Series; return zeros if there is no variation."""
    series = pd.Series(series, dtype=float)
    denominator = series.max() - series.min()
    if denominator == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / denominator


csis_rows = []
cluster_month_counts = all_break_dates.groupby("Break month")["Bank"].nunique() if not all_break_dates.empty else pd.Series(dtype=int)
common_months = set(cluster_month_counts[cluster_month_counts >= 2].index)

for bank in banks:
    abs_n = int(absolute_breaks_table.loc[absolute_breaks_table["Bank"] == bank, "Selected number of breaks"].iloc[0])
    sq_n = int(squared_breaks_table.loc[squared_breaks_table["Bank"] == bank, "Selected number of breaks"].iloc[0])
    icss_n = int(icss_table.loc[icss_table["Bank"] == bank, "Number of ICSS variance breaks"].iloc[0])
    validation_n = int(validation_summary_table.loc[validation_summary_table["Bank"] == bank, "Total significant adjacent-regime differences"].iloc[0])

    bank_breaks = all_break_dates[all_break_dates["Bank"] == bank]
    total_breaks = len(bank_breaks)
    common_breaks = bank_breaks[bank_breaks["Break month"].isin(common_months)].shape[0]
    clustering_share = common_breaks / total_breaks if total_breaks > 0 else 0

    csis_rows.append({
        "Bank": bank,
        "Absolute-return breaks": abs_n,
        "Squared-return breaks": sq_n,
        "ICSS variance breaks": icss_n,
        "Validated adjacent-regime comparisons": validation_n,
        "Clustering proportion": clustering_share
    })

csis_table = pd.DataFrame(csis_rows)

components = {
    "A-score": "Absolute-return breaks",
    "S-score": "Squared-return breaks",
    "V-score": "ICSS variance breaks",
    "R-score": "Validated adjacent-regime comparisons",
    "C-score": "Clustering proportion"
}

for score_col, raw_col in components.items():
    csis_table[score_col] = minmax_scale(csis_table[raw_col])

score_cols = list(components.keys())
csis_table["CSIS"] = csis_table[score_cols].mean(axis=1)
csis_table["Rank"] = csis_table["CSIS"].rank(ascending=False, method="dense").astype(int)

csis_table[score_cols + ["CSIS", "Clustering proportion"]] = csis_table[score_cols + ["CSIS", "Clustering proportion"]].round(3)
csis_table = csis_table.sort_values(["Rank", "Bank"]).reset_index(drop=True)

print("Composite Structural Instability Score:")
display(csis_table)

save_table(csis_table, "Table_7_composite_structural_instability_score.xlsx")

In [ ]:
# Plot compact structural-break counts

break_count_data = csis_table.set_index("Bank")[[
    "Absolute-return breaks",
    "Squared-return breaks",
    "ICSS variance breaks"
]]

ax = break_count_data.plot(kind="bar", figsize=(10, 5))

plt.title("Structural Break Counts across Big Five South African Banking Equities")
plt.xlabel("Bank")
plt.ylabel("Number of breaks")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.legend()

save_figure("Figure_3_structural_break_counts.png")

In [ ]:
# Save final workbook

final_workbook = outputs_path / "paper1_structural_instability_results.xlsx"

with pd.ExcelWriter(final_workbook, engine="openpyxl") as writer:
    summary_table.to_excel(writer, sheet_name="Table1_Descriptives", index=False)
    absolute_breaks_table.to_excel(writer, sheet_name="Table2_AbsBreaks", index=False)
    squared_breaks_table.to_excel(writer, sheet_name="Table3_SqBreaks", index=False)
    icss_table.to_excel(writer, sheet_name="Table4_ICSS", index=False)
    validation_summary_table.to_excel(writer, sheet_name="Table5_RegimeValidation", index=False)
    monthly_break_clusters.to_excel(writer, sheet_name="Table6_BreakClusters", index=False)
    csis_table.to_excel(writer, sheet_name="Table7_CSIS", index=False)
    mean_breaks_table.to_excel(writer, sheet_name="Appendix_MeanBreakCheck", index=False)
    absolute_bic_table.to_excel(writer, sheet_name="Appendix_AbsBIC", index=False)
    squared_bic_table.to_excel(writer, sheet_name="Appendix_SqBIC", index=False)
    regime_validation_table.to_excel(writer, sheet_name="Appendix_RegimeTests", index=False)
    all_break_dates.to_excel(writer, sheet_name="Appendix_AllBreakDates", index=False)

print("Final workbook saved:")
print(final_workbook)

In [ ]:
# Final confirmation

print("Paper 1 analysis completed successfully.")
print("
Outputs saved in:")
print(outputs_path)
print("
Figures saved in:")
print(figures_path)